In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz

# NOTE: Requires 'googleplaystore_user_reviews.csv' in your directory for sentiment data
df_apps = pd.read_csv('googleplaystore.csv')
df_reviews = pd.read_csv('googleplaystore_user_reviews.csv')

df_apps['Installs'] = pd.to_numeric(df_apps['Installs'].astype(str).str.replace(r'[+,]', '', regex=True), errors='coerce')
df_apps['Reviews'] = pd.to_numeric(df_apps['Reviews'], errors='coerce')
df_apps['Rating'] = pd.to_numeric(df_apps['Rating'], errors='coerce')

def parse_size(size_str):
    if pd.isna(size_str) or size_str == 'Varies with device': return np.nan
    if 'M' in size_str: return float(size_str.replace('M', ''))
    if 'k' in size_str: return float(size_str.replace('k', '')) / 1024
    return np.nan
df_apps['Size_MB'] = df_apps['Size'].apply(parse_size)

df_reviews['Sentiment_Subjectivity'] = pd.to_numeric(df_reviews['Sentiment_Subjectivity'], errors='coerce')
subj_df = df_reviews.groupby('App')['Sentiment_Subjectivity'].mean().reset_index()

merged_df = pd.merge(df_apps, subj_df, on='App', how='inner')
merged_df['Category'] = merged_df['Category'].str.upper()

target_cats = ['GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION', 'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENTS']

mask = (
    (merged_df['Rating'] > 3.5) &
    (merged_df['Category'].isin(target_cats)) &
    (merged_df['Reviews'] > 500) &
    (merged_df['Installs'] > 50000) &
    (~merged_df['App'].str.lower().str.contains('s', na=False)) &
    (merged_df['Sentiment_Subjectivity'] > 0.5)
)
final_df = merged_df[mask].copy()

translation_map = {'BEAUTY': 'सौंदर्य', 'BUSINESS': 'வணிகம்', 'DATING': 'Partnersuche'}
final_df['Category'] = final_df['Category'].replace(translation_map)

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()
start_time = datetime.strptime("17:00", "%H:%M").time()
end_time = datetime.strptime("19:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    fig = px.scatter(
        final_df, 
        x='Size_MB', 
        y='Rating', 
        size='Installs', 
        color='Category', 
        hover_name='App',
        color_discrete_map={'GAME': 'pink'},
        title='App Size vs Rating (Bubble Size = Installs)'
    )
    fig.show()
else:
    print(f"Chart hidden. Current IST Time: {current_time.strftime('%I:%M %p')}.")

Chart hidden. Current IST Time: 07:23 PM.
